In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

## Question 1: Install Spark and PySpark

In [2]:
spark = SparkSession.builder \
	.master('local[*]') \
	.appName('test') \
	.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/13 02:44:19 WARN Utils: Your hostname, Kamals-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.11 instead (on interface en0)
26/03/13 02:44:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/13 02:44:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.version

'4.1.1'

## Question 2: Yellow November 2025

In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet -O data/yellow_tripdata_2025-11.parquet

--2026-03-13 01:13:17--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.164.82.197, 3.164.82.40, 3.164.82.160, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.164.82.197|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘data/yellow_tripdata_2025-11.parquet’

data/yellow_tripdat 100%[===================>]  67.84M  2.83MB/s    in 23s     

2026-03-13 01:13:41 (2.90 MB/s) - ‘data/yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [4]:
trips = spark.read.parquet('data/yellow_tripdata_2025-11.parquet')

In [5]:
trips.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [6]:
trips.repartition(4).write.parquet('data/yellow/2025/11', mode='overwrite')

In [7]:
# print average file size in the output directory
!ls -lhF data/yellow/2025/11

total 205312
-rw-r--r--  1 Kamal  staff     0B Mar 13 02:44 _SUCCESS
-rw-r--r--  1 Kamal  staff    24M Mar 13 02:44 part-00000-cb1f3259-3b68-4767-a413-5fffd227df62-c000.snappy.parquet
-rw-r--r--  1 Kamal  staff    24M Mar 13 02:44 part-00001-cb1f3259-3b68-4767-a413-5fffd227df62-c000.snappy.parquet
-rw-r--r--  1 Kamal  staff    24M Mar 13 02:44 part-00002-cb1f3259-3b68-4767-a413-5fffd227df62-c000.snappy.parquet
-rw-r--r--  1 Kamal  staff    24M Mar 13 02:44 part-00003-cb1f3259-3b68-4767-a413-5fffd227df62-c000.snappy.parquet


## Question 3: Count records

In [8]:
trips.filter((trips.tpep_pickup_datetime >= '2025-11-15') & (trips.tpep_pickup_datetime < '2025-11-16')).count()

162604

## Question 4: Longest trip

In [9]:
trips.drop('trip_duration')

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double]

In [10]:
trips = trips.withColumn(
  	'trip_duration', 
	F.round((F.unix_timestamp(trips.tpep_dropoff_datetime) - F.unix_timestamp(trips.tpep_pickup_datetime)) / 3600, 3)
)

In [11]:
trips \
  .select('tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_duration') \
  .orderBy('trip_duration', ascending=False) \
  .limit(5) \
  .show()

+--------------------+---------------------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration|
+--------------------+---------------------+-------------+
| 2025-11-26 20:22:12|  2025-11-30 15:01:00|       90.647|
| 2025-11-27 04:22:41|  2025-11-30 09:19:35|       76.948|
| 2025-11-03 10:42:55|  2025-11-06 14:55:45|       76.214|
| 2025-11-07 11:23:22|  2025-11-10 08:40:41|       69.289|
| 2025-11-18 17:12:47|  2025-11-21 12:17:37|       67.081|
+--------------------+---------------------+-------------+



## Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

- 4040

## Question 6: Least frequent pickup location zone

In [62]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv -O data/taxi_zone_lookup.csv

--2026-03-13 02:13:54--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.164.82.40, 3.164.82.112, 3.164.82.160, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.164.82.40|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘data/taxi_zone_lookup.csv’

data/taxi_zone_look 100%[===================>]  12.04K  --.-KB/s    in 0.1s    

2026-03-13 02:13:57 (117 KB/s) - ‘data/taxi_zone_lookup.csv’ saved [12331/12331]



In [12]:
zones = spark.read.csv('data/taxi_zone_lookup.csv', header=True, inferSchema=True)

In [13]:
zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [14]:
trips.registerTempTable('trips')
zones.registerTempTable('zones')

/Users/Kamal/WorkSpace/Data-Engineering/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [15]:
spark.sql('''
	SELECT 
        zones.Zone,
        trips_count.location_count
    FROM 
        zones
	JOIN (
    	SELECT
			COUNT(*) AS location_count,
			PULocationID AS pickup_location_id
        FROM
			trips
        GROUP BY PULocationID
    ) AS trips_count
    ON trips_count.pickup_location_id = zones.LocationId
    ORDER BY trips_count.location_count ASC
    LIMIT 5
''').head(5)

[Row(Zone="Governor's Island/Ellis Island/Liberty Island", location_count=1),
 Row(Zone="Eltingville/Annadale/Prince's Bay", location_count=1),
 Row(Zone='Arden Heights', location_count=1),
 Row(Zone='Port Richmond', location_count=3),
 Row(Zone='Green-Wood Cemetery', location_count=4)]